# 4. Microsoft GraphRAG

[GraphRAG](https://arxiv.org/abs/2404.16130) builds its index in four stages:

1. **Extract** entities and relations from each chunk, as LightRAG does.
2. **Cluster** the graph into nested communities with the Leiden algorithm.
3. **Summarise** each community into a report written by the LLM.
4. **Embed** entity descriptions, chunks and reports.

Two search modes then use this index:

- **Local search** starts from the entities closest to the question and gathers their relations, source chunks and community reports.
- **Global search** sends batches of community reports to the LLM (map), then merges the partial answers (reduce). It is designed for broad, corpus-wide questions.

In [1]:
from dotenv import load_dotenv

from src import config

load_dotenv(config.PROJECT_ROOT / ".env")
print(f"Domain: {config.DOMAIN} | run mode: {config.RUN_MODE.value}")

Domain: technology | run mode: subset


In [2]:
from src.data import load_full_corpus_statistics, load_run_inputs
from src.usage_tracking import UsageLedger, get_ledger_path, register_litellm_usage_callback

INDEX_NAME = "graphrag"

run_inputs = load_run_inputs(config.DOMAIN, config.RUN_MODE)
ledger = UsageLedger(get_ledger_path(run_inputs.run_directory))
index_directory = run_inputs.run_directory / "indexes" / INDEX_NAME
print(f"{len(run_inputs.questions)} questions, {len(run_inputs.documents)} documents")

9 questions, 23 documents


## 4.1 Configure GraphRAG

GraphRAG reads `graphrag_project/settings.yaml` and its prompts in `graphrag_project/prompts/`. The helper below points the configuration at the documents of this run and applies the shared settings of `src/config.py`.

GraphRAG calls models through LiteLLM. Registering a LiteLLM callback sends the token usage of every call to the ledger.

In [3]:
from src.graphrag_utils import load_graphrag_config

graphrag_config = load_graphrag_config(corpus_directory=run_inputs.corpus_directory, index_directory=index_directory)
register_litellm_usage_callback(ledger)

completion_model = graphrag_config.completion_models["default_completion_model"]
print(f"Model: {completion_model.model} {completion_model.call_args}")
print(f"Chunks: {graphrag_config.chunking.size} tokens, overlap {graphrag_config.chunking.overlap}")
print(f"Index written to: {graphrag_config.output_storage.base_dir}")

Model: gpt-4o-mini {'temperature': 0.0}
Chunks: 1200 tokens, overlap 100
Index written to: /Users/linafaik/Documents/projects/graph-retrieval-bench/outputs/technology/subset/indexes/graphrag/output


## 4.2 Build the index

`build_index` runs the whole pipeline and returns one result per workflow. GraphRAG caches every LLM response in `index_directory/cache`, so an interrupted build resumes without paying twice. Deleting `index_directory` forces a full rebuild.

In [4]:
import time

from graphrag.api import build_index
from graphrag.config.enums import IndexingMethod

from src.usage_tracking import Phase, load_indexing_report, save_indexing_report, usage_scope

if load_indexing_report(run_inputs.run_directory, INDEX_NAME):
    print("Index found, reusing it.")
else:
    if not index_directory.exists():
        ledger.discard_records(INDEX_NAME, Phase.INDEXING)
    start_time = time.perf_counter()
    with usage_scope(INDEX_NAME, Phase.INDEXING):
        workflow_results = await build_index(config=graphrag_config, method=IndexingMethod.Standard)
    for workflow_result in workflow_results:
        print(f"{workflow_result.workflow:<28} {'error: ' + str(workflow_result.error) if workflow_result.error else 'ok'}")
    if any(workflow_result.error for workflow_result in workflow_results):
        raise RuntimeError("GraphRAG indexing failed; see the workflow errors above and the logs folder.")
    save_indexing_report(
        run_inputs.run_directory,
        INDEX_NAME,
        indexing_time_seconds=time.perf_counter() - start_time,
        document_count=len(run_inputs.documents),
        corpus_token_count=int(run_inputs.documents["token_count"].sum()),
    )

indexing_cost_usd = ledger.total_cost_usd(INDEX_NAME, Phase.INDEXING)
print(f"Indexing cost: ${indexing_cost_usd:.4f}")

[2026-09-17T14:42:31Z WARN  lance::dataset::write::insert] No existing dataset at /Users/linafaik/Documents/projects/graph-retrieval-bench/outputs/technology/subset/indexes/graphrag/output/lancedb/entity_description.lance, it will be created
[2026-09-17T14:43:49Z WARN  lance::dataset::write::insert] No existing dataset at /Users/linafaik/Documents/projects/graph-retrieval-bench/outputs/technology/subset/indexes/graphrag/output/lancedb/community_full_content.lance, it will be created
[2026-09-17T14:44:09Z WARN  lance::dataset::write::insert] No existing dataset at /Users/linafaik/Documents/projects/graph-retrieval-bench/outputs/technology/subset/indexes/graphrag/output/lancedb/text_unit_text.lance, it will be created


load_input_documents         ok
create_base_text_units       ok
create_final_documents       ok
extract_graph                ok
finalize_graph               ok
extract_covariates           ok
create_communities           ok
create_final_text_units      ok
create_community_reports     ok
generate_text_embeddings     ok
Indexing cost: $0.3254


In [5]:
from src.usage_tracking import project_full_run_cost_usd

if config.RUN_MODE is config.RunMode.SUBSET:
    projected_cost_usd = project_full_run_cost_usd(
        subset_cost_usd=indexing_cost_usd,
        subset_token_count=int(run_inputs.documents["token_count"].sum()),
        full_token_count=load_full_corpus_statistics(config.DOMAIN)["token_count"],
    )
    print(f"Projected GraphRAG indexing cost on the full corpus: ${projected_cost_usd:.2f}")

Projected GraphRAG indexing cost on the full corpus: $6.55


## 4.3 Look at the index

The index is a set of tables. Entities and relations form the graph; communities group entities; community reports summarise each group.

In [6]:
from src.graphrag_utils import load_graphrag_index_tables

index_tables = await load_graphrag_index_tables(graphrag_config)
{table_name: len(table) for table_name, table in index_tables.items()}

{'entities': 1078,
 'relationships': 994,
 'communities': 112,
 'community_reports': 112,
 'text_units': 160}

In [7]:
index_tables["entities"].sort_values("degree", ascending=False)[["title", "type", "degree", "description"]].head(10)

,title,type,degree,description
0,STEAM,ORGANIZATION,146,Steam is a digital distribution platform for v...
4,VALVE,ORGANIZATION,91,Valve Corporation is a prominent video game de...
496,BLOOMBERG,ORGANIZATION,59,Bloomberg is a prominent global financial serv...
114,PC GAMER,ORGANIZATION,42,PC Gamer is a prominent digital publication th...
266,COUNTER STRIKE GLOBAL OFFENSIVE,ORGANIZATION,30,Counter Strike Global Offensive (CS:GO) is a w...
604,BLOOMBERG L.P.,ORGANIZATION,27,"Bloomberg L.P. is a global financial services,..."
651,BLOOMBERG.COM,ORGANIZATION,26,Bloomberg.com is the main website for Bloomber...
338,EUROGAMER,ORGANIZATION,22,Eurogamer is a prominent gaming news and revie...
713,GEEKWIRE,ORGANIZATION,21,GEEKWIRE is a prominent technology news websit...
243,JOYSTIQ,ORGANIZATION,21,Joystiq is a comprehensive gaming blog and new...


Communities are nested: level 0 holds a few large groups, deeper levels split them further. Search uses the reports up to `GRAPHRAG_COMMUNITY_LEVEL`.

In [8]:
community_reports = index_tables["community_reports"]
print(community_reports.groupby("level").size().rename("reports per level").to_string())

largest_report = community_reports.sort_values("size", ascending=False).iloc[0]
print(f"\n{largest_report['title']} (level {largest_report['level']}, {largest_report['size']} entities)")
print(largest_report["summary"])

level
0    18
1    72
2    20
3     2

Steam and Valve Corporation Community (level 0, 118 entities)
The community centers around Steam, a leading digital distribution platform for video games developed by Valve Corporation. This ecosystem includes various entities such as popular games, developers, and journalists who contribute to the platform's dynamics. The relationships among these entities highlight the interconnectedness of game distribution, user engagement, and content regulation.


## 4.4 Local search

Local search needs the full set of tables. It returns the answer and the context it was built from.

In [9]:
from graphrag.api import global_search, local_search

from src.question_runner import answer_all_questions


async def answer_with_local_search(question: str, question_type: str) -> str:
    """Answer one question with GraphRAG local search."""
    response, _ = await local_search(
        config=graphrag_config,
        entities=index_tables["entities"],
        communities=index_tables["communities"],
        community_reports=index_tables["community_reports"],
        text_units=index_tables["text_units"],
        relationships=index_tables["relationships"],
        covariates=None,
        community_level=config.GRAPHRAG_COMMUNITY_LEVEL,
        response_type=config.RESPONSE_TYPE,
        query=question,
    )
    return response


local_search_predictions = await answer_all_questions(
    answer_function=answer_with_local_search,
    questions=run_inputs.questions,
    system_name="graphrag_local",
    run_directory=run_inputs.run_directory,
    max_concurrent_questions=config.MAX_CONCURRENT_QUESTIONS,
)

graphrag_local: 0 answered, 9 to go.


graphrag_local:   0%|          | 0/9 [00:00<?, ?it/s]

## 4.5 Global search

Global search only needs entities, communities and reports: it never reads the raw chunks. Dynamic community selection is off, the default of the GraphRAG CLI, so every report up to the chosen level is read.

In [10]:
async def answer_with_global_search(question: str, question_type: str) -> str:
    """Answer one question with GraphRAG global search."""
    response, _ = await global_search(
        config=graphrag_config,
        entities=index_tables["entities"],
        communities=index_tables["communities"],
        community_reports=index_tables["community_reports"],
        community_level=config.GRAPHRAG_COMMUNITY_LEVEL,
        dynamic_community_selection=False,
        response_type=config.RESPONSE_TYPE,
        query=question,
    )
    return response


global_search_predictions = await answer_all_questions(
    answer_function=answer_with_global_search,
    questions=run_inputs.questions,
    system_name="graphrag_global",
    run_directory=run_inputs.run_directory,
    max_concurrent_questions=config.MAX_CONCURRENT_QUESTIONS,
)

graphrag_global: 0 answered, 9 to go.


graphrag_global:   0%|          | 0/9 [00:00<?, ?it/s]

## 4.6 One question, two search modes

In [11]:
example_question_id = run_inputs.questions.loc[run_inputs.questions["question_type"] == "summary", "question_id"].iloc[0]
example_question = run_inputs.questions.set_index("question_id").loc[example_question_id]
print("Q:", example_question["question"], "\n")
for system_name, predictions in [("graphrag_local", local_search_predictions), ("graphrag_global", global_search_predictions)]:
    answer_text = predictions.set_index("question_id").loc[example_question_id, "pred_answer"]
    print(f"--- {system_name}\n{answer_text[:700]}\n")

Q: How has Valve approached quality control and disallowed functionality, particularly in response to developers submitting manipulative or bad-faith games? 

--- graphrag_local
## Valve's Approach to Quality Control and Disallowed Functionality

Valve Corporation, a leading entity in the gaming industry, has implemented several measures to maintain quality control on its digital distribution platform, Steam. This is particularly important given the vast number of games submitted by developers, some of which may be manipulative or of poor quality. Valve's strategies focus on content management, community engagement, and evolving policies to ensure a trustworthy gaming environment.

### Content Management Policies

One of the primary ways Valve addresses quality control is through its content management policies. The company has faced scrutiny regarding the presence 

--- graphrag_global
## Valve's Approach to Quality Control and Disallowed Functionality

Valve Corporation has taken sig

## 4.7 What it cost

Both search modes share one index. Their query costs differ: global search makes one map call per batch of reports for every question.

In [12]:
ledger_summary = ledger.summarize_by_system_and_phase()
ledger_summary[ledger_summary["system_name"].isin([INDEX_NAME, "graphrag_local", "graphrag_global"])]

,system_name,phase,model,call_count,prompt_tokens,completion_tokens,cost_usd
0,graphrag,indexing,gpt-4o-mini,720,1445458,259269,0.319378
1,graphrag,indexing,text-embedding-3-small,103,302930,0,0.006059
2,graphrag_global,query,gpt-4o-mini,72,732499,16827,0.078259
3,graphrag_local,query,gpt-4o-mini,9,88684,3633,0.015118
4,graphrag_local,query,text-embedding-3-small,9,264,0,0.000005


## Next

All four systems have answered. Notebook 05 grades the answers and puts quality and cost side by side.